<a href="https://colab.research.google.com/github/goutham3010/Hospital-Readmission-Prediction-System/blob/main/05_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================
# STEP 05: MODEL EVALUATION
# Hospital Readmission Prediction
# ============================================

In [ ]:
# ============================================
# 1. Import Required Libraries
# ============================================
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
# ============================================
# 2. Store Trained Models in a Dictionary
# ============================================
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model
}

In [ ]:
# ============================================
# 3. Function: Evaluate Models & Compute Metrics
# ============================================
def evaluate_models(models_dict, X_test, y_test):
    """
    Evaluates multiple classification models and returns a comparison DataFrame.
    """
    results = []

    for name, model in models_dict.items():
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        results.append({
            "Model": name,
            "Accuracy": round(accuracy, 4),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1 Score": round(f1, 4)
        })

    results_df = pd.DataFrame(results)
    return results_df

# Run Evaluation
results_df = evaluate_models(models, X_test, y_test)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)
display(results_df)

In [ ]:
# ============================================
# 4. Function: Rank Models & Find Best Performer
# ============================================
def get_best_model(results_df, target_metric="F1 Score"):
    """
    Sorts models by a specified metric and identifies the top performing model.
    """
    results_sorted = results_df.sort_values(by=target_metric, ascending=False).reset_index(drop=True)
    best_model_name = results_sorted.iloc[0]["Model"]
    best_score = results_sorted.iloc[0][target_metric]

    print(f"MODELS SORTED BY {target_metric.upper()}")
    print("=" * 70)
    display(results_sorted)

    print("\n" + "=" * 70)
    print(f"🏆 BEST MODEL (by {target_metric}): {best_model_name} ({target_metric}: {best_score:.4f})")
    print("=" * 70)

    return results_sorted, best_model_name

# Find best model
results_sorted, best_model_name = get_best_model(results_df, target_metric="F1 Score")

In [ ]:
# ============================================
# 5. Function: Display Classification Reports
# ============================================
def display_classification_reports(models_dict, X_test, y_test, target_names=None):
    """
    Prints the detailed classification report for each model.
    """
    if target_names is None:
        target_names = ["Not Readmitted (0)", "Readmitted (1)"]

    for name, model in models_dict.items():
        y_pred = model.predict(X_test)

        print("\n" + "=" * 70)
        print(f"CLASSIFICATION REPORT: {name}")
        print("=" * 70)
        print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

# Display reports
display_classification_reports(models, X_test, y_test)

In [ ]:
# ============================================
# 6. Function: Plot Confusion Matrices
# ============================================
def plot_confusion_matrices(models_dict, X_test, y_test, display_labels=None):
    """
    Generates and displays confusion matrices for all models side-by-side.
    """
    if display_labels is None:
        display_labels = ["Not Readmitted (0)", "Readmitted (1)"]

    num_models = len(models_dict)
    fig, axes = plt.subplots(1, num_models, figsize=(5 * num_models, 4))

    if num_models == 1:
        axes = [axes]

    for ax, (name, model) in zip(axes, models_dict.items()):
        y_pred = model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)

        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
        disp.plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(f"Confusion Matrix\n{name}")

    plt.tight_layout()
    plt.show()

# Plot confusion matrices
plot_confusion_matrices(models, X_test, y_test)

In [ ]:
# ============================================
# 7. Function: Plot Model Performance Comparison
# ============================================
def plot_model_comparison(results_df):
    """
    Plots a grouped bar chart comparing all metrics across models.
    """
    results_plot = results_df.set_index("Model")

    ax = results_plot.plot(
        kind="bar",
        figsize=(10, 6),
        colormap="viridis",
        edgecolor="black"
    )

    plt.title("Model Performance Comparison Across Metrics", fontsize=14, fontweight="bold", pad=15)
    plt.ylabel("Score", fontsize=12)
    plt.xlabel("Model", fontsize=12)
    plt.xticks(rotation=0, fontsize=11)
    plt.ylim(0, 1.1)
    plt.legend(title="Metrics", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

# Plot comparison chart
plot_model_comparison(results_df)

In [ ]:
# ============================================
# 8. Final Model Evaluation Summary
# ============================================
print("=" * 70)
print("FINAL MODEL EVALUATION SUMMARY")
print("=" * 70)

display(results_sorted)

print(f"\nRecommended Model for Deployment: {best_model_name}")